In [1]:
import numpy as np
from typing import Dict
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import torch.nn.functional as F
import math

In [2]:
dropout = 0.1
lr=1e-3
weight_decay= 1e-4
batch_size = 256
epochs  = 20 
seed  = 24
device = "cuda:1" if torch.cuda.is_available() else "cpu"

In [3]:
import pandas as pd
data=pd.read_csv("../example_data/pred_hsATAC.csv")

In [4]:
data

,gene,gene_log2FC,loop_log2FC,K4_log2FC,K27_log2FC,atac_ckmean,atac_hsmean,com_log2FC
0,Zm00001d020750,-0.316372,-0.205038,2.094062,0.610108,-0.069284,-0.056039,-0.658424
1,Zm00001d007947,0.301064,-0.292364,0.219568,-0.087086,-0.186460,-0.170282,0.101372
2,Zm00001d038248,-0.185700,0.201547,0.434437,0.438035,0.021330,-0.049925,-1.112589
3,Zm00001d038249,-0.210810,0.378074,-0.099209,-1.255572,0.024450,0.020907,-1.112589
4,Zm00001d013244,1.411458,0.384275,-1.079375,-0.058806,0.170977,0.042714,-1.015229
...,...,...,...,...,...,...,...,...
21407,Zm00001d026149,-0.032332,0.799154,0.000000,0.000000,-0.017010,-0.028464,-2.062067
21408,Zm00001d044514,0.419259,0.466276,0.329968,0.393220,-0.161105,-0.132449,-0.543382
21409,Zm00001d026144,-0.163029,0.997068,0.000000,0.000000,-0.115210,-0.129248,-0.488484
21410,Zm00001d026147,-2.081324,0.755835,0.000000,0.000000,-0.046262,-0.076978,-2.054824


create dataloader

In [5]:
def split_indices(n: int, seed: int = seed) -> Dict[str, np.ndarray]:
    """
    train_idx(64%) / loop_idx(16%) / test_idx(20%)
    """
    rng = np.random.default_rng(seed)
    all_idx = np.arange(n)
    rng.shuffle(all_idx)
    n_train_zone = int(0.8 * n)
    train_zone = all_idx[:n_train_zone]    # 80%
    test_idx   = all_idx[n_train_zone:]    # 20%
    n_train = int(0.8 * train_zone.size)   # 80% of 80% = 64%
    train_idx = train_zone[:n_train]
    loop_idx  = train_zone[n_train:]       # 20% of 80% = 16%
    return dict(train=train_idx, loop=loop_idx, test=test_idx)

In [6]:
splits = split_indices(len(data),seed=seed)
tr_idx, loop_idx, te_idx = splits["train"], splits["loop"], splits["test"]

In [7]:
class multiDataset(Dataset):
    def __init__(self,mat,feature,label):
        self.gene=mat["gene"].values
        self.mat=torch.as_tensor(mat[feature].values,dtype=torch.float32)
        self.label=torch.as_tensor(mat[label].values,dtype=torch.float32)
    def __len__(self):
        return len(self.label)

    def __getitem__(self,i):
        return self.mat[i],self.label[i],self.gene[i]


In [8]:
train_data=data.loc[tr_idx]
valid_data=data.loc[loop_idx]
test_data=data.loc[te_idx]

In [9]:
feature_cols = [c for c in data.columns if c not in ["gene",  "atac_hsmean"]]
ds_train=multiDataset(train_data,feature_cols, "atac_hsmean") 
dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)

ds_valid=multiDataset(valid_data,feature_cols, "atac_hsmean") 
dl_valid = DataLoader(ds_valid, batch_size=batch_size, shuffle=True)

ds_test=multiDataset(test_data,feature_cols, "atac_hsmean") 
dl_test = DataLoader(ds_test, batch_size=batch_size, shuffle=True)

In [10]:
for ma,la,gene in dl_train:
    print(ma.shape)
    print(la.shape)
    print(len(gene))
    break

torch.Size([256, 6])
torch.Size([256])
256


main model

In [11]:
class net(nn.Module):
    def __init__(self,dropout):
        super().__init__()
        self.stru=nn.Sequential(
            nn.Linear(6,32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32,16),
            nn.ReLU(),
            nn.Linear(16,1)
        )
    def forward(self,mat):
        value=self.stru(mat)
        return value

train

In [12]:
def calc_pearson(la_true, la_pred):
    la_true = np.asarray(la_true).reshape(-1)
    la_pred = np.asarray(la_pred).reshape(-1)

    if len(la_true) < 2:
        return np.nan
    if np.std(la_true) == 0 or np.std(la_pred) == 0:
        return np.nan

    return np.corrcoef(la_true,la_pred)[0, 1]

In [13]:
def run_one_epoch(model,loader,optimizer,device,train=True):
    model.train(mode=train)
    total_loss=0.0
    total_sq_error=0.0
    total_abs_error=0.0
    total = 0
    all_preds=[]
    all_labels=[]
    for data,label,_ in loader:
        data=data.to(device)
        label=label.to(device).view(-1, 1)
        pred=model(data)
        loss = F.smooth_l1_loss(pred,label)
        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        batchSize = label.size(0)
        diff=pred-label

    
        total_loss += loss.item() * batchSize
        total_sq_error += torch.sum(diff ** 2).item()
        total_abs_error += torch.sum(torch.abs(diff)).item()
        total += batchSize

        all_preds.append(pred.detach().cpu().numpy())
        all_labels.append(label.detach().cpu().numpy())
    avg_loss = total_loss / max(total, 1)
    rmse = math.sqrt(total_sq_error / max(total, 1))
    mae = total_abs_error / max(total, 1)

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    pearson = calc_pearson(all_labels, all_preds)
    return avg_loss, rmse, mae, pearson

In [14]:
def train_model(model,dl_train,dl_valid,device,to):
    torch.manual_seed(seed); np.random.seed(seed)
    device=torch.device(device)
    optim=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=weight_decay)
    
    best_path = to
    best_rmse = 100
    best_loss=100
    min_diff=1e-3
    no_improve=0
    num_no_improve = 5
    ep_idx=0

    f_train_loss=0
    f_train_rmse=0
    f_train_mae=0
    f_train_pearson=0
    f_val_loss=0
    f_val_rmse=0
    f_val_mae=0
    f_val_pearson=0

    for ep in range(1,epochs+1):
        train_loss, train_rmse, train_mae, train_pearson = run_one_epoch(model, dl_train, optim, device, train=True)
        val_loss, val_rmse, val_mae, val_pearson = run_one_epoch(model, dl_valid,optimizer=None, device=device, train=False)
        
        print(f"[Epoch_{ep:02d}] "
            f"train_loss {train_loss:.4f} rmse {train_rmse:.4f} "
            f"train_mae {train_mae:.4f} train_pearson {train_pearson:.4f} "
            f"valid_loss {val_loss:.4f} rmse {val_rmse:.4f} "
            f"valid_mae {val_mae:.4f} val_pearson {val_pearson:.4f} ")
    
        if val_rmse  < best_rmse:
            best_rmse=val_rmse
            torch.save(model.state_dict(), best_path)
            ep_idx=ep
            f_train_loss=train_loss
            f_train_rmse=train_rmse
            f_train_mae=train_mae
            f_train_pearson=train_pearson
            f_val_loss=val_loss
            f_val_rmse=val_rmse
            f_val_mae=val_mae
            f_val_pearson=val_pearson

        if best_loss - val_loss > min_diff:
            best_loss = val_loss
            no_improve = 0
        else:
            no_improve +=1
            if no_improve > num_no_improve:
                print(f"Early stopping at epoch {ep}")
                break
    print(f"final saved model at {ep_idx}")
    print(f"[Epoch_{ep_idx:02d}] "
            f"train_loss {f_train_loss:.4f} rmse {f_train_rmse:.4f} "
            f"train_mae {f_train_mae:.4f} train_pearson {f_train_pearson:.4f} "
            f"valid_loss {f_val_loss:.4f} rmse {f_val_rmse:.4f} "
            f"valid_mae {f_val_mae:.4f} val_pearson {f_val_pearson:.4f} ")
    return f_train_loss,f_train_rmse,f_train_mae,f_train_pearson,f_val_loss,f_val_rmse,f_val_mae,f_val_pearson

In [15]:
model=net(dropout=dropout).to(device)

In [16]:
f_train_loss,f_train_rmse,f_train_mae,f_train_pearson,f_val_loss,f_val_rmse,f_val_mae,f_val_pearson = \
train_model(model,dl_train,dl_valid,device,to="best_model_3layernet_ATAC.pt")

[Epoch_01] train_loss 0.0068 rmse 0.1165 train_mae 0.0894 train_pearson 0.0182 valid_loss 0.0050 rmse 0.0997 valid_mae 0.0759 val_pearson 0.1738 
[Epoch_02] train_loss 0.0046 rmse 0.0961 train_mae 0.0733 train_pearson 0.2110 valid_loss 0.0044 rmse 0.0940 valid_mae 0.0711 val_pearson 0.3901 
[Epoch_03] train_loss 0.0041 rmse 0.0908 train_mae 0.0694 train_pearson 0.3649 valid_loss 0.0039 rmse 0.0879 valid_mae 0.0671 val_pearson 0.5665 
[Epoch_04] train_loss 0.0036 rmse 0.0853 train_mae 0.0652 train_pearson 0.4925 valid_loss 0.0032 rmse 0.0794 valid_mae 0.0595 val_pearson 0.7198 
[Epoch_05] train_loss 0.0029 rmse 0.0763 train_mae 0.0581 train_pearson 0.6306 valid_loss 0.0021 rmse 0.0651 valid_mae 0.0497 val_pearson 0.8123 
[Epoch_06] train_loss 0.0023 rmse 0.0674 train_mae 0.0511 train_pearson 0.7236 valid_loss 0.0015 rmse 0.0554 valid_mae 0.0419 val_pearson 0.8511 
[Epoch_07] train_loss 0.0019 rmse 0.0614 train_mae 0.0462 train_pearson 0.7767 valid_loss 0.0013 rmse 0.0513 valid_mae 0.038

validation on test data

In [17]:
def gather_pred(model,loader,device):
    model.eval()
    total_loss = 0.0
    total_sq_error = 0.0
    total_abs_error = 0.0
    total = 0
    all_preds = []
    all_labels = []
    all_gene = []

    with torch.no_grad():
        for data, label, gene in loader:
            data = data.to(device)
            label = label.to(device).float().view(-1, 1)

            pred = model(data)
            loss = F.smooth_l1_loss(pred, label)

            diff = pred - label
            batch_size = label.size(0)

            total_loss += loss.item() * batch_size
            total_sq_error += torch.sum(diff ** 2).item()
            total_abs_error += torch.sum(torch.abs(diff)).item()
            total += batch_size

            all_preds.append(pred.cpu().numpy())
            all_labels.append(label.cpu().numpy())
            all_gene.extend(gene)


    avg_loss = total_loss / max(total, 1)
    rmse = math.sqrt(total_sq_error / max(total, 1))
    mae = total_abs_error / max(total, 1)

    all_preds = np.concatenate(all_preds, axis=0).reshape(-1)
    all_labels = np.concatenate(all_labels, axis=0).reshape(-1)
    pearson = calc_pearson(all_labels, all_preds)

    return avg_loss, rmse, mae, pearson,all_preds,all_labels,all_gene

In [18]:
avg_loss, rmse, mae, pearson,all_preds,all_labels,all_gene=gather_pred(model,dl_test,device)

In [19]:
print(f"avg_loss:{avg_loss}")
print(f"rmse:{rmse}")
print(f"mae:{mae}")
print(f"pearson:{pearson}")

avg_loss:0.0009562690888165979
rmse:0.0437325756764411
mae:0.032397379976495694
pearson:0.8964303433102407


In [20]:
test_data={"gene":all_gene,
           "preds":all_preds,
           "label":all_labels
}
test_data2=pd.DataFrame(test_data)

In [21]:
test_data2.to_csv("testdata_predresult.csv",index=False)